# Reading from the silver table/s

In [0]:
df_sls = spark.read.table("workspace.silver.crm_sales_details")
df_cust = spark.read.table("workspace.gold.dim_customers")
df_prd = spark.read.table("workspace.gold.dim_products")


# Creating temporary view

In [0]:
# Register them as Temporary Views so Spark SQL can see them
df_sls.createOrReplaceTempView("s")
df_cust.createOrReplaceTempView("c")
df_prd.createOrReplaceTempView("p")


# Create customers dimension table

In [0]:
query = """
    SELECT
        s.sales_order_number,
        c.customer_surrogate_key as customer_key,
        p.product_surrogate_key as product_key,
        s.sales_order_date,
        s.sales_shiping_date,
        s.sales_due_date,
        s.total_sales,
        s.sales_quantity,
        s.sales_price
    FROM s
    LEFT JOIN c
    on s.sales_customer_id = c.customer_id
    LEFT JOIN p
    on s.sales_product_key = p.sales_product_key
"""

df = spark.sql(query)


# Write to Gold

In [0]:
(
    df.write
        .mode("overwrite")
        .format("delta")
        .saveAsTable("gold.fact_sales")
)

In [0]:
%sql
select * from workspace.gold.fact_sales